In [13]:
import warnings
import zarr
from obstore.store import from_url

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

zarr.config.set({"async.concurrency": 4})

bucket = "s3://noaa-cdr-patmosx-radiances-and-clouds-pds"
store  = from_url(bucket, region="us-east-1", skip_signature=True)

# List only the 1981 folder
objects_1981 = list(store.list(prefix="data/1981/"))

# Inspect what one object actually looks like
print(type(objects_1981[0]))
print(objects_1981[0])


<class 'list'>
[{'path': 'data/1981/patmosx_v06r00_NOAA-07_asc_d19810825_c20221117.nc', 'last_modified': datetime.datetime(2026, 1, 12, 9, 54, 59, tzinfo=datetime.timezone.utc), 'size': 566678828, 'e_tag': '"7bdac5d3a24b49661b45b20d0589dcc3-11"', 'version': None}, {'path': 'data/1981/patmosx_v06r00_NOAA-07_asc_d19810826_c20221117.nc', 'last_modified': datetime.datetime(2026, 1, 12, 10, 2, 44, tzinfo=datetime.timezone.utc), 'size': 548673121, 'e_tag': '"f319ba815a7a54b56a7a69c27ff3cbb9-11"', 'version': None}, {'path': 'data/1981/patmosx_v06r00_NOAA-07_asc_d19810827_c20221117.nc', 'last_modified': datetime.datetime(2026, 1, 12, 10, 4, tzinfo=datetime.timezone.utc), 'size': 551697904, 'e_tag': '"79558ed538d10c302d373dab6b239767-11"', 'version': None}, {'path': 'data/1981/patmosx_v06r00_NOAA-07_asc_d19810828_c20221117.nc', 'last_modified': datetime.datetime(2026, 1, 12, 9, 59, 6, tzinfo=datetime.timezone.utc), 'size': 549818457, 'e_tag': '"65a7bd1f9447e5b866c0d7b2d771c7d9-11"', 'version': 

In [14]:
import warnings
import shutil
from pathlib import Path

import zarr
import xarray as xr
import icechunk
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import HDFParser

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

zarr.config.set({"async.concurrency": 4})

# ── 1. BUCKET & STORE ─────────────────────────────────────────────────────────
bucket = "s3://noaa-cdr-patmosx-radiances-and-clouds-pds"
store  = from_url(bucket, region="us-east-1", skip_signature=True)


In [15]:
# ── 2. COLLECT 1981 URLs FROM BUCKET LISTING ──────────────────────────────────
# obstore returns an iterator of *lists* (batches), not individual objects
# We need to flatten the batches first
objects_1981 = [
    obj
    for batch in store.list(prefix="data/1981/")
    for obj in batch
]

urls_1981 = sorted([
    f"{bucket}/{obj['path']}"
    for obj in objects_1981
    if obj['path'].endswith('.nc')
])

print(f"Files found in 1981: {len(urls_1981)}")


Files found in 1981: 202


In [16]:
# Check what satellite/node combinations exist in 1981
import re
from collections import Counter

# Extract the satellite+node part from each filename
# Pattern: patmosx_v06r00_{SAT}_{NODE}_d{DATE}_c{CREATED}.nc
patterns = []
for url in urls_1981:
    match = re.search(r'patmosx_v06r00_(.+?)_d\d{8}', url)
    if match:
        patterns.append(match.group(1))

counts = Counter(patterns)
for key, count in sorted(counts.items()):
    print(f"  {key:30s}  {count} files")

  NOAA-07_asc                     101 files
  NOAA-07_des                     101 files


In [17]:
urls_1981_asc = [u for u in urls_1981 if '_asc_' in u]
print(f"Files after filtering to asc: {len(urls_1981_asc)}")  # expect 101

# Quick sanity check — print first and last
print("First:", urls_1981_asc[0])
print("Last: ", urls_1981_asc[-1])

Files after filtering to asc: 101
First: s3://noaa-cdr-patmosx-radiances-and-clouds-pds/data/1981/patmosx_v06r00_NOAA-07_asc_d19810825_c20221117.nc
Last:  s3://noaa-cdr-patmosx-radiances-and-clouds-pds/data/1981/patmosx_v06r00_NOAA-07_asc_d19811231_c20221117.nc


In [18]:
import re

# Extract dates for asc and des separately
asc_dates = set()
des_dates = set()

for url in urls_1981:
    # Match pattern: patmosx_v06r00_{SAT}_{NODE}_d{DATE}_c{CREATED}.nc
    match = re.search(r'_d(\d{8})_', url)
    if match:
        date = match.group(1)
        if '_asc_' in url:
            asc_dates.add(date)
        elif '_des_' in url:
            des_dates.add(date)

# Dates that have BOTH asc and des
both_dates = sorted(asc_dates & des_dates)

# Print results
print(f"Total dates with ascending passes: {len(asc_dates)}")
print(f"Total dates with descending passes: {len(des_dates)}")
print(f"Dates with both asc and des: {len(both_dates)}")
print(both_dates)

Total dates with ascending passes: 101
Total dates with descending passes: 101
Dates with both asc and des: 101
['19810825', '19810826', '19810827', '19810828', '19810829', '19810830', '19810831', '19810901', '19810902', '19810904', '19810907', '19810908', '19810912', '19810913', '19810914', '19810915', '19810916', '19810917', '19810918', '19810919', '19810920', '19810921', '19810922', '19810923', '19810924', '19810925', '19810926', '19810927', '19810928', '19810929', '19810930', '19811001', '19811003', '19811006', '19811007', '19811008', '19811009', '19811010', '19811011', '19811012', '19811016', '19811017', '19811019', '19811020', '19811021', '19811023', '19811024', '19811025', '19811026', '19811029', '19811030', '19811031', '19811101', '19811102', '19811103', '19811104', '19811105', '19811107', '19811108', '19811109', '19811110', '19811111', '19811112', '19811113', '19811114', '19811115', '19811116', '19811117', '19811118', '19811119', '19811120', '19811121', '19811123', '19811126',

In [ ]:
# ── 3. OPEN VIRTUAL MULTI-FILE DATASET ────────────────────────────────────────
registry = ObjectStoreRegistry({bucket: store})
parser   = HDFParser()

combined_vds = open_virtual_mfdataset(
    urls_1981_asc,
    registry=registry,
    parser=parser,
    combine="nested",
    concat_dim="time",
    combine_attrs="drop_conflicts",
    loadable_variables=["time", "latitude", "longitude"],
    decode_times=True,
)

print(combined_vds)
print(f"Time steps: {combined_vds.dims['time']}")  # expect 101

# Sanity check — should show 101 time steps, and 3 variables of interest
print(combined_vds["cloud_fraction"])
print(combined_vds["cld_opd_dcomp"])
print(combined_vds["cld_press_acha"])


/home/coder/envs/protocoast-notebook/lib/python3.12/site-packages/virtualizarr/xarray.py:463: FutureWarning: In a future version, xarray will not decode the variable 'scan_line_time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  with xr.open_zarr(


In [ ]:
import json

# Check dataset-level attrs
for k, v in combined_vds.attrs.items():
    try:
        json.dumps(v)
    except (TypeError, ValueError) as e:
        print(f"Dataset attr '{k}': {type(v).__name__} = {repr(v)[:80]}  → {e}")

# Check variable attrs
for var in list(combined_vds.data_vars) + list(combined_vds.coords):
    for k, v in combined_vds[var].attrs.items():
        try:
            json.dumps(v)
        except (TypeError, ValueError) as e:
            print(f"  [{var}] attr '{k}': {type(v).__name__} = {repr(v)[:80]}  → {e}")

In [11]:
# ── 4. WRITE VIRTUAL REFERENCES TO ICECHUNK ───────────────────────────────────
repo_path = Path("patmosx-icechunk-1981")
if repo_path.exists():
    shutil.rmtree(repo_path)
    print(f"Cleared existing repo at {repo_path}/")

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix="s3://noaa-cdr-patmosx-radiances-and-clouds-pds/",
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    ),
)

storage = icechunk.local_filesystem_storage(str(repo_path))
repo    = icechunk.Repository.create(storage, config)

session = repo.writable_session("main")
combined_vds.vz.to_icechunk(session.store)
snapshot_id = session.commit("Add PATMOS-x 1981 (101 days, NOAA-07 asc)")
print("Committed:", snapshot_id)


  2026-03-24T19:05:08.680499Z  WARN icechunk::storage::object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk/src/storage/object_store.rs:92



IcechunkError:   x bad metadata
  | 
  | context:
  |    0: icechunk::store::set
  |            with key="zarr.json"
  |              at icechunk/src/store.rs:287
  | 
  |-> bad metadata
  `-> expected value at line 74 column 30


In [ ]:
# ── 5. READ BACK & VERIFY ─────────────────────────────────────────────────────
credentials = icechunk.containers_credentials({
    "s3://noaa-cdr-patmosx-radiances-and-clouds-pds/":
        icechunk.s3_credentials(anonymous=True)
})

repo2   = icechunk.Repository.open(
    storage, config, authorize_virtual_chunk_access=credentials
)
session = repo2.readonly_session("main")

ds = xr.open_zarr(session.store, consolidated=False, chunks=None)
print(ds)

# Plot global mean cloud fraction across all 49 days
ds["cloud_fraction"].mean(["latitude", "longitude"]).plot(marker="o")

# Map of cloud-top pressure for the first available day (1981-08-25)
ds["cld_press_acha"].isel(time=0).plot()